PySpark Transformations Deep Dive

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

events = spark.read.csv(
    "/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv",
    header=True,
    inferSchema=True
)


events.count()


42448764

In [0]:
products = events.select(
    "product_id", "brand", "category_code"
).dropDuplicates()


In [0]:
joined = events.join(products, on="product_id", how="inner")
joined.show(5)


+----------+-------------------+----------+-------------------+--------------------+------+------+---------+--------------------+------+--------------------+
|product_id|         event_time|event_type|        category_id|       category_code| brand| price|  user_id|        user_session| brand|       category_code|
+----------+-------------------+----------+-------------------+--------------------+------+------+---------+--------------------+------+--------------------+
|   1005159|2019-10-01 02:21:26|      view|2053013555631882655|electronics.smart...|xiaomi|231.41|516846425|5e9060c8-a2b1-4dc...|xiaomi|electronics.smart...|
|   1005159|2019-10-01 02:21:54|      view|2053013555631882655|electronics.smart...|xiaomi|231.41|530803522|8a23287a-5b5a-4a9...|xiaomi|electronics.smart...|
|   1005159|2019-10-01 02:36:20|      view|2053013555631882655|electronics.smart...|xiaomi|231.41|545434640|93236651-68a6-4f7...|xiaomi|electronics.smart...|
|   5701087|2019-10-01 02:38:37|      view|205301355

In [0]:
events.join(products, on="product_id", how="left").count()


44180379

In [0]:
window = Window.partitionBy("user_id").orderBy("event_time")

events_with_running = events.withColumn(
    "cumulative_events",
    F.count("*").over(window)
)

events_with_running.select(
    "user_id", "event_time", "cumulative_events"
).show(10)


+---------+-------------------+-----------------+
|  user_id|         event_time|cumulative_events|
+---------+-------------------+-----------------+
|205053188|2019-10-09 10:30:19|                1|
|205053188|2019-10-09 10:30:44|                2|
|209714031|2019-10-20 18:29:45|                1|
|209714031|2019-10-20 18:30:08|                2|
|209714031|2019-10-24 18:18:25|                3|
|209714031|2019-10-24 18:27:59|                4|
|209714031|2019-10-24 18:29:31|                5|
|209714031|2019-10-25 06:59:08|                6|
|209714031|2019-10-29 19:39:37|                7|
|209714031|2019-10-29 19:41:10|                8|
+---------+-------------------+-----------------+
only showing top 10 rows


In [0]:
purchase_value = (
    events
    .filter(F.col("event_type") == "purchase")
    .groupBy("user_id")
    .agg(F.sum("price").alias("total_spent"))
)

rank_window = Window.orderBy(F.desc("total_spent"))

ranked_users = purchase_value.withColumn(
    "rank", F.dense_rank().over(rank_window)
)

ranked_users.show(10)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+------------------+----+
|  user_id|       total_spent|rank|
+---------+------------------+----+
|519267944| 265569.5200000001|   1|
|513117637|244499.99999999994|   2|
|515384420|210749.77000000002|   3|
|530834332|187128.93000000005|   4|
|512386086|          182470.8|   5|
|519250600|174449.34999999998|   6|
|513784794|         143821.03|   7|
|538216048|         134344.69|   8|
|514320330|119103.98999999998|   9|
|553431815|118762.24000000002|  10|
+---------+------------------+----+
only showing top 10 rows


In [0]:
top_products = (
    events
    .filter(F.col("event_type") == "purchase")
    .groupBy("product_id")
    .agg(F.round(F.sum("price"), 2).alias("revenue"))
    .orderBy(F.desc("revenue"))
    .limit(5)
)

top_products.show()


+----------+-------------+
|product_id|      revenue|
+----------+-------------+
|   1005115|1.240680735E7|
|   1005105|1.023924868E7|
|   1004249|   6730112.92|
|   1005135|   5567806.64|
|   1004767|   5430723.43|
+----------+-------------+



In [0]:
conversion = (
    events
    .groupBy("category_code", "event_type")
    .count()
    .groupBy("category_code")
    .pivot("event_type")
    .sum("count")
    .withColumn(
        "conversion_rate",
        F.round((F.col("purchase") / F.col("view")) * 100, 2)
    )
)

conversion.show(10)


+--------------------+-----+--------+-------+---------------+
|       category_code| cart|purchase|   view|conversion_rate|
+--------------------+-----+--------+-------+---------------+
|auto.accessories....| NULL|      46|  12305|           0.37|
|furniture.living_...| NULL|    1084| 215471|            0.5|
| stationery.cartrige|  106|     134|   7380|           1.82|
|       sport.bicycle|  693|     838| 128759|           0.65|
|        apparel.sock|    7|      21|   2621|            0.8|
|appliances.enviro...|   16|      27|   2172|           1.24|
|          kids.swing|  147|     330|  31596|           1.04|
|electronics.audio...|  196|     430|  28394|           1.51|
|auto.accessories....|  716|     494|  42350|           1.17|
|  electronics.clocks|20344|   17906|1272783|           1.41|
+--------------------+-----+--------+-------+---------------+
only showing top 10 rows


In [0]:
features = events.withColumn(
    "is_purchase",
    F.when(F.col("event_type") == "purchase", 1).otherwise(0)
)

features.select("event_type", "is_purchase").show(5)


+----------+-----------+
|event_type|is_purchase|
+----------+-----------+
|      view|          0|
|      view|          0|
|      view|          0|
|      view|          0|
|      view|          0|
+----------+-----------+
only showing top 5 rows


In [0]:
from pyspark.sql.types import StringType

def price_bucket(price):
    if price < 50:
        return "low"
    elif price < 200:
        return "medium"
    else:
        return "high"

bucket_udf = F.udf(price_bucket, StringType())

events.withColumn("price_category", bucket_udf("price")).show(5)


+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+--------------+
|         event_time|event_type|product_id|        category_id|       category_code|   brand|  price|  user_id|        user_session|price_category|
+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+--------------+
|2019-10-01 00:00:00|      view|  44600062|2103807459595387724|                NULL|shiseido|  35.79|541312140|72d76fde-8bb3-4e0...|           low|
|2019-10-01 00:00:00|      view|   3900821|2053013552326770905|appliances.enviro...|    aqua|   33.2|554748717|9333dfbd-b87a-470...|           low|
|2019-10-01 00:00:01|      view|  17200506|2053013559792632471|furniture.living_...|    NULL|  543.1|519107250|566511c2-e2e3-422...|          high|
|2019-10-01 00:00:01|      view|   1307067|2053013558920217191|  computers.notebook|  lenovo| 251.74|550050854|7